# Environment Setup - E-Commerce Agent Workshop

This notebook prepares the AWS account and local notebook environment for the workshop.

You will learn how the workshop keeps setup state durable across notebooks, how the sample data is loaded, and how to verify that the resources needed by later agent, evaluation, and observability steps are ready. By the end of this notebook, you should have a working AWS sandbox, sample product data, and a local state file that later notebooks can read.

## Step 1: Resolve Notebook Paths

Different editors start Jupyter kernels from different working directories. This cell finds the workshop root and Section 00 folder explicitly so every later file access is based on stable paths.

After running it, check the printed paths. If they point to the workspace repo, the notebook can find setup scripts, sample data, and shared state files reliably.

In [ ]:
from pathlib import Path
import sys


def _find_section_dir():
    start = Path.cwd().resolve()
    for parent in [start, *start.parents]:
        for candidate in (parent, parent / "00-prerequisites"):
            if (candidate / "sample_data" / "orders.json").is_file() and (
                candidate / "setup_infrastructure.py"
            ).is_file():
                return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate 00-prerequisites. Open this notebook from the workshop repo root "
        "or the 00-prerequisites folder."
    )


SECTION_DIR = _find_section_dir()
if str(SECTION_DIR) not in sys.path:
    sys.path.insert(0, str(SECTION_DIR))

from workshop_paths import read_sample_json, section_file_path

print(f"Workshop section directory: {SECTION_DIR}")

## Step 2: Provision AWS Infrastructure

This cell creates the baseline AWS resources used by the workshop and loads the starter product data.

The important learning point is resource ownership: the setup script writes the identifiers it creates into workshop state, so later notebooks do not need to rediscover or hard-code table names. Read the output for created or reused resources and for any permission errors that need attention before continuing.

In [ ]:
# Detect AWS region from environment or use default
import boto3

session = boto3.Session()
AWS_REGION = session.region_name or 'us-west-2'
print(f"Using AWS Region: {AWS_REGION}")

In [ ]:
# Run infrastructure setup
import subprocess

subprocess.run(
    [sys.executable, str(section_file_path("setup_infrastructure.py", section_dir=SECTION_DIR)), "--region", AWS_REGION],
    check=True,
)

## Step 3: Verify Infrastructure

This cell checks whether the required resources exist and whether optional convenience settings are available.

Treat this as the workshop health check. After running it, inspect the pass/fail summary. Required checks should pass before continuing; optional checks explain which convenience features are available or skipped.


In [ ]:
# Verify infrastructure is ready
import subprocess

subprocess.run(
    [sys.executable, str(section_file_path("verify_infrastructure.py", section_dir=SECTION_DIR))],
    check=True,
)

## Step 4: Inspect The Sample Data

This cell loads the local sample data used to seed the product catalog.

Before building an agent, look at the product categories, product IDs, inventory fields, and account/order examples. These records shape the questions the agent can answer and the evaluation cases you will see later.

In [ ]:
import textwrap
import pandas as pd
from IPython.display import display

# Load sample data from local files (no AWS connection needed)
orders_data = read_sample_json("orders.json", section_dir=SECTION_DIR)
accounts_data = read_sample_json("accounts.json", section_dir=SECTION_DIR)
products_data = read_sample_json("products.json", section_dir=SECTION_DIR)

print("Sample data loaded successfully!")
print(f"  - {len(accounts_data['accounts'])} customer accounts")
print(f"  - {len(orders_data['orders'])} orders")
print(f"  - {len(products_data['products'])} products in the catalog")


In [ ]:

# Display customer accounts with membership tier breakdown
accounts_df = pd.DataFrame([
    {
        "customer_id": a["customer_id"],
        "name": f"{a['first_name']} {a['last_name']}",
        "membership_tier": a["membership_tier"],
        "account_status": a["account_status"],
        "total_orders": a["total_orders"],
        "total_spent": f"${a['total_spent']:,.2f}",
        "member_since": a["created_date"],
    }
    for a in accounts_data["accounts"]
])

print("=== Customer Accounts ===")
display(accounts_df)

print("\n--- Membership Tier Distribution ---")
tier_counts = accounts_df["membership_tier"].value_counts().rename("count").to_frame()
display(tier_counts)

print("\n--- Account Status Distribution ---")
status_counts = accounts_df["account_status"].value_counts().rename("count").to_frame()
display(status_counts)


In [ ]:

# Display orders with status breakdown
orders_df = pd.DataFrame([
    {
        "order_id": o["order_id"],
        "customer_id": o["customer_id"],
        "status": o["status"],
        "order_date": o["order_date"],
        "items": len(o["items"]),
        "total": f"${o['total']:,.2f}",
    }
    for o in orders_data["orders"]
])

print("=== Orders ===")
display(orders_df)

print("\n--- Order Status Distribution ---")
print("(These represent the different customer service scenarios agents will handle)")
status_counts = orders_df["status"].value_counts().rename("count").to_frame()
display(status_counts)

print("\n--- Orders per Customer ---")
orders_per_customer = (
    orders_df.groupby("customer_id")
    .agg(num_orders=("order_id", "count"))
    .sort_values("num_orders", ascending=False)
)
display(orders_per_customer)


In [ ]:

# Display product catalog and store policies
products_df = pd.DataFrame([
    {
        "product_id": p["product_id"],
        "name": p["name"],
        "category": p["category"],
        "price": f"${p['price']:.2f}",
        "in_stock": "Yes" if p["in_stock"] else "No",
        "stock_qty": p["stock_quantity"],
        "rating": p["rating"],
        "warranty": p["warranty"],
    }
    for p in products_data["products"]
])

print("=== Product Catalog ===")
display(products_df)

print("\n--- Products by Category ---")
category_summary = (
    products_df.assign(price_raw=[p["price"] for p in products_data["products"]])
    .groupby("category")
    .agg(count=("product_id", "count"), avg_price=("price_raw", "mean"))
    .rename(columns={"count": "# Products", "avg_price": "Avg Price ($)"})
    .round(2)
)
display(category_summary)

print("\n=== Store Policies ===")
for key, policy_text in products_data["policies"].items():
    print(f"\n{key.replace('_', ' ').title()}:")
    # Word-wrap at ~100 chars
    import textwrap
    for line in textwrap.wrap(policy_text, width=100):
        print(f"  {line}")


## Setup Complete

At this point, you should know three things:

1. Where the workshop root and state files are.
2. Which AWS resources were created or reused.
3. What product data the agent will use.

Continue when the verification output is clean. The next notebook builds the local Product Catalog Agent and tests its role-based behavior.